In [1]:
from pyspark.sql import functions as F
from data_generator import get_spark, generate_large_table, generate_small_lookup
spark = get_spark("Case07_CatalystOptimizer")

df=generate_large_table(spark,num_rows=5_000_000)
lookup = generate_small_lookup(spark, num_rows=100)

26/04/14 13:53:13 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


# Demo 1: Predicate Pushdown

In [2]:
print("=" * 60)
print("DEMO 1: Predicate Pushdown")
print("=" * 60)

df.write.mode("overwrite").parquet("/tmp/case07_events")
result = (
    spark.read.parquet("/tmp/case07_events")
    .filter(F.col("event_date") == "2025-01-15")
    .filter(F.col("download_bytes") > 500000)
    .select("account_number", "download_bytes")
)
print("\nPredicate Pushdown Plan:")
result.explain(True)

DEMO 1: Predicate Pushdown


26/04/14 13:53:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/14 13:53:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/14 13:53:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/14 13:53:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/14 13:53:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/14 13:53:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/04/14 13:53:16 WARN MemoryManager: Total allocation exceeds 95.


Predicate Pushdown Plan:
== Parsed Logical Plan ==
'Project ['account_number, 'download_bytes]
+- Filter (download_bytes#60L > cast(500000 as bigint))
   +- Filter (event_date#58 = cast(2025-01-15 as date))
      +- Relation [id#56L,account_number#57,event_date#58,session_duration#59,download_bytes#60L,upload_bytes#61L,category#62] parquet

== Analyzed Logical Plan ==
account_number: string, download_bytes: bigint
Project [account_number#57, download_bytes#60L]
+- Filter (download_bytes#60L > cast(500000 as bigint))
   +- Filter (event_date#58 = cast(2025-01-15 as date))
      +- Relation [id#56L,account_number#57,event_date#58,session_duration#59,download_bytes#60L,upload_bytes#61L,category#62] parquet

== Optimized Logical Plan ==
Project [account_number#57, download_bytes#60L]
+- Filter ((isnotnull(event_date#58) AND isnotnull(download_bytes#60L)) AND ((event_date#58 = 2025-01-15) AND (download_bytes#60L > 500000)))
   +- Relation [id#56L,account_number#57,event_date#58,session_dur

# Demo 2: Projection Pruning (Column Pruning)

In [3]:
print("=" * 60)
print("DEMO 2: Projection Pruning")
print("=" * 60)

result2=(
    spark.read.parquet("/tmp/case07_events")
    .select("account_number","download_bytes")
    .filter(F.col("download_bytes")>100000)
)
print("\nColumn Pruning Plan:")
result2.explain(True)

DEMO 2: Projection Pruning

Column Pruning Plan:
== Parsed Logical Plan ==
'Filter ('download_bytes > 100000)
+- Project [account_number#74, download_bytes#77L]
   +- Relation [id#73L,account_number#74,event_date#75,session_duration#76,download_bytes#77L,upload_bytes#78L,category#79] parquet

== Analyzed Logical Plan ==
account_number: string, download_bytes: bigint
Filter (download_bytes#77L > cast(100000 as bigint))
+- Project [account_number#74, download_bytes#77L]
   +- Relation [id#73L,account_number#74,event_date#75,session_duration#76,download_bytes#77L,upload_bytes#78L,category#79] parquet

== Optimized Logical Plan ==
Project [account_number#74, download_bytes#77L]
+- Filter (isnotnull(download_bytes#77L) AND (download_bytes#77L > 100000))
   +- Relation [id#73L,account_number#74,event_date#75,session_duration#76,download_bytes#77L,upload_bytes#78L,category#79] parquet

== Physical Plan ==
*(1) Filter (isnotnull(download_bytes#77L) AND (download_bytes#77L > 100000))
+- *(1) Co

# Demo 3: Constant Folding

In [5]:
print("=" * 60)
print("DEMO 3: Constant Folding")
print("=" * 60)

result3 = df.filter(F.col("download_bytes") > F.lit(1000) * F.lit(500))
print("\nConstant Folding Plan:")
result3.explain(True)


DEMO 3: Constant Folding

Constant Folding Plan:
== Parsed Logical Plan ==
'Filter ('download_bytes > (1000 * 500))
+- Project [id#0L, account_number#2, event_date#5, session_duration#9, download_bytes#14L, upload_bytes#20L, concat(cat_, cast((id#0L % cast(50 as bigint)) as string)) AS category#27]
   +- Project [id#0L, account_number#2, event_date#5, session_duration#9, download_bytes#14L, cast((rand(-2590738588708193137) * cast(500000 as double)) as bigint) AS upload_bytes#20L]
      +- Project [id#0L, account_number#2, event_date#5, session_duration#9, cast((rand(-4024410629741481258) * cast(1000000 as double)) as bigint) AS download_bytes#14L]
         +- Project [id#0L, account_number#2, event_date#5, cast((rand(7480067263117318486) * cast(3600 as double)) as int) AS session_duration#9]
            +- Project [id#0L, account_number#2, date_add(cast(2025-01-01 as date), cast((id#0L % cast(90 as bigint)) as int)) AS event_date#5]
               +- Project [id#0L, cast((id#0L % cast(

# Demo 4: Join Reordering

In [6]:
print("=" * 60)
print("DEMO 4: Auto Broadcast Detection")
print("=" * 60)
result4=df.join(lookup,"category","inner")
print("\n Auto Broadcast Plan:")
result4.explain(True)

DEMO 4: Auto Broadcast Detection

 Auto Broadcast Plan:
== Parsed Logical Plan ==
'Join UsingJoin(Inner, [category])
:- Project [id#0L, account_number#2, event_date#5, session_duration#9, download_bytes#14L, upload_bytes#20L, concat(cat_, cast((id#0L % cast(50 as bigint)) as string)) AS category#27]
:  +- Project [id#0L, account_number#2, event_date#5, session_duration#9, download_bytes#14L, cast((rand(-2590738588708193137) * cast(500000 as double)) as bigint) AS upload_bytes#20L]
:     +- Project [id#0L, account_number#2, event_date#5, session_duration#9, cast((rand(-4024410629741481258) * cast(1000000 as double)) as bigint) AS download_bytes#14L]
:        +- Project [id#0L, account_number#2, event_date#5, cast((rand(7480067263117318486) * cast(3600 as double)) as int) AS session_duration#9]
:           +- Project [id#0L, account_number#2, date_add(cast(2025-01-01 as date), cast((id#0L % cast(90 as bigint)) as int)) AS event_date#5]
:              +- Project [id#0L, cast((id#0L % cast

# Demo 5: What Blocks Catalyst?

In [7]:
print("=" * 60)
print("DEMO 5: Things That Block Optimization")
print("=" * 60)
from pyspark.sql.types import BooleanType
bad_filter=F.udf(lambda x:x>500000, BooleanType())
result_bad=(
    spark.read.parquet("/tmp/case07_events")
    .filter(bad_filter(F.col("download_bytes")))
)
print("\n UDF blocks pushdown:")
result_bad.explain(True)

DEMO 5: Things That Block Optimization

 UDF blocks pushdown:
== Parsed Logical Plan ==
'Filter <lambda>('download_bytes)#114
+- Relation [id#100L,account_number#101,event_date#102,session_duration#103,download_bytes#104L,upload_bytes#105L,category#106] parquet

== Analyzed Logical Plan ==
id: bigint, account_number: string, event_date: date, session_duration: int, download_bytes: bigint, upload_bytes: bigint, category: string
Filter <lambda>(download_bytes#104L)#114
+- Relation [id#100L,account_number#101,event_date#102,session_duration#103,download_bytes#104L,upload_bytes#105L,category#106] parquet

== Optimized Logical Plan ==
Project [id#100L, account_number#101, event_date#102, session_duration#103, download_bytes#104L, upload_bytes#105L, category#106]
+- Filter pythonUDF0#116: boolean
   +- BatchEvalPython [<lambda>(download_bytes#104L)#114], [pythonUDF0#116]
      +- Relation [id#100L,account_number#101,event_date#102,session_duration#103,download_bytes#104L,upload_bytes#105L,ca

In [8]:
result_good=(
    spark.read.parquet("/tmp/case07_events")
    .filter(F.col("download_bytes") > 500000)
)
print("\n Built-in function allows pushdown:")
result_good.explain(True)


 Built-in function allows pushdown:
== Parsed Logical Plan ==
'Filter ('download_bytes > 500000)
+- Relation [id#117L,account_number#118,event_date#119,session_duration#120,download_bytes#121L,upload_bytes#122L,category#123] parquet

== Analyzed Logical Plan ==
id: bigint, account_number: string, event_date: date, session_duration: int, download_bytes: bigint, upload_bytes: bigint, category: string
Filter (download_bytes#121L > cast(500000 as bigint))
+- Relation [id#117L,account_number#118,event_date#119,session_duration#120,download_bytes#121L,upload_bytes#122L,category#123] parquet

== Optimized Logical Plan ==
Filter (isnotnull(download_bytes#121L) AND (download_bytes#121L > 500000))
+- Relation [id#117L,account_number#118,event_date#119,session_duration#120,download_bytes#121L,upload_bytes#122L,category#123] parquet

== Physical Plan ==
*(1) Filter (isnotnull(download_bytes#121L) AND (download_bytes#121L > 500000))
+- *(1) ColumnarToRow
   +- FileScan parquet [id#117L,account_num